# Blind screening

Try removing the name, pronouns and university from the resume text before scoring. See if the score change goes down.

Heads up: in our data the only difference between the original and a counterfactual is exactly the field we delete. So if we delete it from both, the texts become the same and the score difference must be 0. We talk about this in the report.

In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import os, pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
# upload jobs.csv and resume_variants.csv
from google.colab import files
up = files.upload()
for f in up:
    os.rename(f, f"data/{f}")

In [ ]:
jobs = pd.read_csv("data/jobs.csv")
res = pd.read_csv("data/resume_variants.csv")

In [ ]:
# strip out name, university, and pronoun words (including loose ones inside sentences)
pron_words = ["she/her","he/him","they/them"," she "," he "," they "," her "," him "," them "," their "]

def blind(row):
    t = str(row["resume_text"])
    t = t.replace(str(row["name"]), "")
    t = t.replace(str(row["university"]), "")
    for w in pron_words:
        t = t.replace(w, " ")
    return " ".join(t.split())

res["blind_text"] = res.apply(blind, axis=1)
res[["resume_id","version","blind_text"]].head()

In [ ]:
jobs["job_text"] = jobs["title"] + " " + jobs["domain"] + " " + jobs["company_name"] + " " + jobs["job_description"]

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
job_emb = model.encode(jobs["job_text"].tolist())
blind_emb = model.encode(res["blind_text"].tolist())

In [ ]:
out = []
for i in range(len(res)):
    for j in range(len(jobs)):
        s = float(cosine_similarity(blind_emb[i:i+1], job_emb[j:j+1])[0][0])
        out.append({
            "resume_id": res.iloc[i]["resume_id"],
            "version": res.iloc[i]["version"],
            "changed_signal": res.iloc[i]["changed_signal"],
            "job_id": jobs.iloc[j]["job_id"],
            "job_title": jobs.iloc[j]["title"],
            "blind_similarity_score": s,
        })
bs = pd.DataFrame(out)
print(len(bs))

In [ ]:
orig = bs[bs.version=="original"][["resume_id","job_id","job_title","blind_similarity_score"]]
orig = orig.rename(columns={"blind_similarity_score":"blind_original_score"})
ch = bs[bs.version!="original"].rename(columns={"blind_similarity_score":"blind_changed_score"})

cmp = ch.merge(orig, on=["resume_id","job_id","job_title"], how="left")
cmp["blind_score_difference"] = cmp["blind_changed_score"] - cmp["blind_original_score"]
cmp["blind_absolute_difference"] = cmp["blind_score_difference"].abs()
cmp.head()

In [ ]:
summary = cmp.groupby("changed_signal").agg(
    average_blind_score_difference=("blind_score_difference","mean"),
    average_blind_absolute_difference=("blind_absolute_difference","mean"),
    max_blind_absolute_difference=("blind_absolute_difference","max"),
    min_blind_score_difference=("blind_score_difference","min"),
    max_blind_score_difference=("blind_score_difference","max"),
).reset_index()
summary

All zeros, as expected. Once you delete the only thing that was different, the two resumes are the same text and so they have the same embedding.

In [ ]:
bs.to_csv("results/blind_screening_scores.csv", index=False)
cmp.to_csv("results/blind_screening_comparison.csv", index=False)
summary.to_csv("results/blind_screening_summary.csv", index=False)
for f in ["blind_screening_scores.csv","blind_screening_comparison.csv","blind_screening_summary.csv"]:
    files.download(f"results/{f}")